# KDAL V21 1 PM Bucket Challenger

Bucket-first probability challenger layered on the immutable V20 KDAL 1 PM no-peak point model. It uses strict nested forward folds, a dedicated `kdal_1pm` feature profile, and an untouched 2026 report. A successful training run is not automatically promotable; the historical acceptance gates must also pass.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'calibration' / 'bucket_probability.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not locate weather-research project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
POINT_DIR = PROJECT_ROOT / 'data/calibration/station_stacking_v20_kdal_1pm_no_peak'
OUTPUT_DIR = PROJECT_ROOT / 'data/calibration/station_stacking_v21_kdal_1pm_bucket'
POINT_VERSION = 'station_high_regressor_v20_kdal_1pm_no_peak_stack'
BUCKET_VERSION = 'station_bucket_v21_kdal_1pm'
RUN_TRAINING = False
PROJECT_ROOT


## Source contract


In [ ]:
point_manifest_path = POINT_DIR / 'model_weights' / f'KDAL_{POINT_VERSION}.json'
point_manifest = json.loads(point_manifest_path.read_text(encoding='utf-8'))
contract = point_manifest['model_contract']
assert contract['timing_mode'] == 'same_day_1pm_live_safe'
assert contract['feature_version'] == 'v20_kdal_1pm_no_peak'
assert contract['target_mode'] == 'remaining_warmup'
assert contract['target_source'] == 'wunderground_only'
contract


## Train the challenger


In [ ]:
bucket_manifest_path = OUTPUT_DIR / 'model_weights' / f'KDAL_{BUCKET_VERSION}.json'
if RUN_TRAINING or not bucket_manifest_path.exists():
    command = [
        str(PYTHON), str(PROJECT_ROOT / 'scripts/train-bucket-probability.py'),
        '--station', 'KDAL', '--pipeline-dir', str(POINT_DIR),
        '--point-bundle', str(POINT_DIR / 'model_weights' / f'KDAL_{POINT_VERSION}.joblib'),
        '--point-model-version', POINT_VERSION, '--model-version', BUCKET_VERSION,
        '--feature-profile', 'kdal_1pm', '--output-dir', str(OUTPUT_DIR),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Using existing v21 artifacts; set RUN_TRAINING=True to retrain.')


## End-to-end mismatch audit


In [ ]:
subprocess.run(
    [str(PYTHON), str(PROJECT_ROOT / 'scripts/audit_v21_kdal_1pm_bucket.py')],
    cwd=PROJECT_ROOT, check=True,
)
audit = json.loads((OUTPUT_DIR / 'audit/audit_result.json').read_text(encoding='utf-8'))
assert audit['passed']
audit


## Forward and holdout results


In [ ]:
forward = pd.read_csv(OUTPUT_DIR / 'KDAL_forward_probability_metrics.csv')
holdout = pd.read_csv(OUTPUT_DIR / 'KDAL_2026_probability_holdout_metrics.csv')
profile = pd.read_csv(OUTPUT_DIR / 'KDAL_probability_feature_profile_comparison.csv')
display(profile)
display(forward)
display(holdout)


## Promotion decision


In [ ]:
manifest = json.loads(bucket_manifest_path.read_text(encoding='utf-8'))
acceptance = manifest['historical_acceptance']
if acceptance['passed']:
    print('Historical gates passed; shadow evaluation is the next step.')
else:
    print('RESEARCH-ONLY: promotion gates failed:', ', '.join(acceptance['reasons']))
acceptance
